# 02 — Construir corpus Reglamento FIFA 2026

**Proyecto:** Asistente Agentic RAG sobre Mundial 2026
**Curso:** PLN — USFQ

Este notebook parsea el PDF oficial del Reglamento FIFA 2026 (en español), lo divide por artículo / sección y guarda cada sección como `.md` con frontmatter YAML.

**Input:** `Corpus_Mundial/reglamento-fifa/FWC26_regulations_ES.pdf`

**Output:** `.md` por artículo/sección en la misma carpeta `Corpus_Mundial/reglamento-fifa/` (~30–80 archivos).

## 1. Setup

In [1]:
# !pip install pypdf pyyaml

In [2]:
import re
import unicodedata
from pathlib import Path

from pypdf import PdfReader
import yaml

## 2. Configuración

In [3]:
ROOT = Path('..').resolve()
OUTPUT_DIR = ROOT / 'Corpus_Mundial' / 'reglamento-fifa'
PDF_PATH = OUTPUT_DIR / 'FWC26_regulations_ES.pdf'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert PDF_PATH.exists(), f'No se encuentra el PDF en {PDF_PATH}'
print(f'PDF input:   {PDF_PATH}')
print(f'Output dir:  {OUTPUT_DIR}')

PDF input:   C:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\reglamento-fifa\FWC26_regulations_ES.pdf
Output dir:  C:\Users\Administrador\Documents\MaestriaUSFQ\Procesamiento Lenguaje Natural\Proyecto Final\Corpus_Mundial\reglamento-fifa


## 3. Extraer texto del PDF

In [4]:
reader = PdfReader(str(PDF_PATH))
n_pages = len(reader.pages)
print(f'Páginas: {n_pages}')

# Extraer página por página y unir con saltos de línea
pages_text = []
for i, page in enumerate(reader.pages):
    t = page.extract_text() or ''
    pages_text.append(t)

full_text = '\n'.join(pages_text)
print(f'Caracteres extraídos: {len(full_text):,}')

Páginas: 99
Caracteres extraídos: 142,044


## 4. Inspeccionar la estructura

Veamos los primeros 2000 caracteres para entender cómo está organizado el reglamento. Esto nos dice qué patrón usar para dividir (Artículo, Capítulo, Sección, etc.).

In [5]:
print(full_text[:2000])

Reglamento de la 
Copa Mundial  
de la FIFA 26™
11 de junio – 19 de julio de 2026
MAYO DE 2026

Fédération Internationale de Football Association
Presidente:  Gianni Infantino
Secretario general:  Mattias Grafström
Página web: FIFA.com
Comisión Organizadora de Competiciones de la FIFA
Presidente:	 Aleksander 	Čeferin
Índice
3
ÍNDICE
I .
DISPOSICIONES GENERALES  7
Artículo 1. Copa Mundial de la FIFA™ 8
Artículo 2. Fase preliminar 9
Artículo 3. Ente organizador de la FIFA 9
Artículo 4. Filiales locales de la FIFA 10
Artículo 5. Responsabilidades de las federaciones miembro participantes 10
Artículo 6. Retirada, partido no disputado, partido suspendido y sustitución 12
II .
CUESTIONES Y PROCEDIMIENTOS DISCIPLINARIOS  15
Artículo 7. Cuestiones disciplinarias 16
Artículo 8. Resolución de disputas 16
Artículo 9. Protestas 16
Artículo 10. Tarjetas amarillas y rojas 18
III .
FORMATO DE LA COMPETICIÓN 19
Artículo 11. Número de equipos 20
Artículo 12. Fase de grupos y fase de eliminación directa

In [6]:
# Conteo de marcadores comunes en reglamentos FIFA
patrones_inspeccion = {
    'Artículo N':       r'Art[íi]culo\s+\d+',
    'Art. N':           r'Art\.\s*\d+',
    'CAPÍTULO romano':  r'CAP[ÍI]TULO\s+[IVXLCDM]+',
    'Capítulo arábigo': r'Cap[íi]tulo\s+\d+',
    'Sección N':        r'Secci[óo]n\s+\d+',
    'Título N':         r'T[íi]tulo\s+\d+',
}

print('Conteo de marcadores en el documento:')
for nombre, patron in patrones_inspeccion.items():
    matches = re.findall(patron, full_text, flags=re.IGNORECASE)
    print(f'  {nombre:20s}  {len(matches):4d} ocurrencias')

Conteo de marcadores en el documento:
  Artículo N             104 ocurrencias
  Art. N                   5 ocurrencias
  CAPÍTULO romano          0 ocurrencias
  Capítulo arábigo         0 ocurrencias
  Sección N                0 ocurrencias
  Título N                 0 ocurrencias


## 5. Definir el splitter

Basado en el conteo anterior, elige el patrón que mejor segmenta el documento. Por defecto usamos **Artículo N** porque suele dar la granularidad ideal para chunks (1 doc por artículo = ~200–500 palabras).

Si el conteo de "Artículo N" es bajo, prueba con "Sección" o "Capítulo".

In [7]:
# Patrón principal — regex MULTILINE para matchear solo headers reales
# (Artículo N al inicio de línea, no referencias dentro del texto).
SPLIT_PATTERN = r'(?m)(?=^\s*Art[íi]culo\s+\d+)'

# Dividir conservando los headers
secciones_raw = re.split(SPLIT_PATTERN, full_text, flags=re.IGNORECASE)
preambulo = secciones_raw[0].strip() if secciones_raw else ''
secciones_pre = [s.strip() for s in secciones_raw[1:] if s.strip()]

# Dedupe: el TOC al inicio del PDF tambien matchea "Articulo N" pero contiene solo
# el titulo + numero de pagina (muy poco contenido). El articulo real tiene mucho mas.
# Estrategia: agrupar por numero de articulo y conservar la version mas larga.
def _get_articulo_num(text):
    m = re.match(r'\s*Art[íi]culo\s+(\d+)', text, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

dedup = {}
for sec in secciones_pre:
    num = _get_articulo_num(sec)
    if num is None:
        continue
    if num not in dedup or len(sec) > len(dedup[num]):
        dedup[num] = sec

# Reordenar por numero de articulo
secciones = [dedup[k] for k in sorted(dedup.keys())]

print(f'Preámbulo / texto pre-artículos: {len(preambulo):,} chars')
print(f'Secciones detectadas tras split: {len(secciones_pre)}')
print(f'Secciones después de dedupe:     {len(secciones)}')

Preámbulo / texto pre-artículos: 365 chars
Secciones detectadas tras split: 104
Secciones después de dedupe:     52


In [8]:
# Preview: primeras 3 secciones
for i, sec in enumerate(secciones[:3]):
    print(f'--- Sección {i+1} ({len(sec)} chars) ---')
    print(sec[:300])
    print('...')
    print()

--- Sección 1 (2434 chars) ---
ARTíCULO 1. COPA MUNDIAL DE LA FIFA™
1.1 La Copa Mundial de la FIFA™ es una competición recogida en los Estatutos 
de la FIFA. 
1.2 La Copa Mundial de la FIFA™ se celebra cada cuatro años. Por regla general, toda 
federación	 afiliada	a	la	FIFA	puede	participar	 en	la	Copa	Mundial	de	la	FIFA™.
1.3 L
...

--- Sección 2 (1896 chars) ---
ARTíCULO 2. FASE PRELIMINAR
2.1 La FIFA organizará la fase preliminar en colaboración con las confederaciones, 
quienes han establecido un formato de competición que recibió la requerida 
aprobación de la FIFA.
2.2 Al inscribirse en la fase preliminar, las federaciones miembro:
a)	 reconocen	y	acept
...

--- Sección 3 (1308 chars) ---
ARTíCULO 3. ENTE ORGANIZADOR DE LA FIFA
3.1 El Consejo de la FIFA o la comisión pertinente nombrada por el Consejo de la 
FIFA serán responsables de organizar el Mundial de la FIFA 26 de acuerdo con 
el presente reglamento, los Estatutos de la FIFA, el Reglamento de Gobernanza 
de la FIFA y el 

## 6. Helpers

In [9]:
def slugify(text: str) -> str:
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode()
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[\s_]+', '-', text)


def extraer_titulo(seccion_text: str, max_chars: int = 200) -> str:
    """Toma el titulo del articulo, combinando 1-2 lineas si el PDF lo partio."""
    lines = [l.strip() for l in seccion_text.split('\n') if l.strip()]
    if not lines:
        return 'sin-titulo'
    titulo = lines[0]
    # Si la primera linea no termina en punto y hay segunda linea corta, asumir
    # que el titulo continua en la segunda linea (caso comun por wrap del PDF).
    if (not titulo.rstrip().endswith('.')
            and len(titulo) < 100
            and len(lines) > 1
            and not re.match(r'^\d+\.\d+', lines[1])  # no es contenido tipo "5.1"
            and len(lines[1]) < 150):
        titulo = f'{titulo} {lines[1]}'
    return titulo[:max_chars]


def extraer_numero_articulo(titulo: str) -> str:
    """Extrae el numero del articulo del titulo, ej. 'Articulo 5. ...' -> '005'."""
    m = re.search(r'Art[íi]culo\s+(\d+)', titulo, flags=re.IGNORECASE)
    if m:
        return f'{int(m.group(1)):03d}'
    return None


def write_seccion_md(seccion_text: str, idx: int) -> Path:
    titulo = extraer_titulo(seccion_text)
    numero = extraer_numero_articulo(titulo)
    if numero:
        slug = f'articulo-{numero}-{slugify(titulo)[:60]}'
    else:
        slug = f'seccion-{idx:03d}-{slugify(titulo)[:60]}'

    fm = {
        'titulo': titulo,
        'tema': 'reglamento',
        'tipo': 'normativo',
        'fuente': 'FIFA Regulations 2026 (FWC26_regulations_ES.pdf)',
        'articulo': numero,
        'tags': ['fifa', 'reglamento', 'normativo', 'mundial-2026'],
    }
    yaml_block = yaml.safe_dump(fm, allow_unicode=True, sort_keys=False)
    content = f'---\n{yaml_block}---\n\n# {titulo}\n\n{seccion_text}\n'
    out_path = OUTPUT_DIR / f'{slug}.md'
    out_path.write_text(content, encoding='utf-8')
    return out_path

## 7. Escribir secciones como `.md`

In [10]:
# Escribir el preámbulo si tiene contenido relevante (>200 chars)
if len(preambulo) > 200:
    preambulo_path = OUTPUT_DIR / '000-preambulo.md'
    fm = {
        'titulo': 'Preámbulo y disposiciones iniciales',
        'tema': 'reglamento',
        'tipo': 'normativo',
        'fuente': 'FIFA Regulations 2026 (FWC26_regulations_ES.pdf)',
        'tags': ['fifa', 'reglamento', 'preambulo'],
    }
    yaml_block = yaml.safe_dump(fm, allow_unicode=True, sort_keys=False)
    content = f'---\n{yaml_block}---\n\n# Preámbulo y disposiciones iniciales\n\n{preambulo}\n'
    preambulo_path.write_text(content, encoding='utf-8')
    print(f'✓ Preámbulo escrito: {preambulo_path.name}')

✓ Preámbulo escrito: 000-preambulo.md


In [11]:
# Escribir cada sección/artículo
written = []
for i, seccion in enumerate(secciones, start=1):
    if len(seccion) < 50:
        # Sección muy corta, probablemente ruido (header sin contenido)
        continue
    path = write_seccion_md(seccion, idx=i)
    written.append(path)
    print(f'  ✓ {path.name} ({len(seccion)} chars)')

print()
print(f'Total secciones escritas: {len(written)}')

  ✓ articulo-001-articulo-1-copa-mundial-de-la-fifatm.md (2434 chars)
  ✓ articulo-002-articulo-2-fase-preliminar.md (1896 chars)
  ✓ articulo-003-articulo-3-ente-organizador-de-la-fifa.md (1308 chars)
  ✓ articulo-004-articulo-4-filiales-locales-de-la-fifa.md (851 chars)
  ✓ articulo-005-articulo-5-responsabilidades-de-las-federaciones-miembro-par.md (5238 chars)
  ✓ articulo-006-articulo-6-retirada-partido-no-disputado-partido-suspendido-.md (5048 chars)
  ✓ articulo-007-articulo-7-cuestiones-disciplinarias.md (869 chars)
  ✓ articulo-008-articulo-8-resolucion-de-disputas.md (1002 chars)
  ✓ articulo-009-articulo-9-protestas.md (3655 chars)
  ✓ articulo-010-articulo-10-tarjetas-amarillas-y-rojas.md (2237 chars)
  ✓ articulo-011-articulo-11-numero-de-equipos.md (2383 chars)
  ✓ articulo-012-articulo-12-fase-de-grupos-y-fase-de-eliminacion-directa.md (7195 chars)
  ✓ articulo-013-articulo-13-igualdad-de-puntos-y-clasificacion-para-la-fase-.md (3971 chars)
  ✓ articulo-014-articulo-14-p

## 8. Validación

Listar archivos generados y revisar tamaños.

In [12]:
archivos = sorted(OUTPUT_DIR.glob('*.md'))
print(f'Total archivos .md en {OUTPUT_DIR.name}: {len(archivos)}')
print()
print('Primeros 10 archivos:')
for f in archivos[:10]:
    print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

if len(archivos) > 10:
    print(f'  ... y {len(archivos) - 10} más')

Total archivos .md en reglamento-fifa: 53

Primeros 10 archivos:
  000-preambulo.md  (630 bytes)
  articulo-001-articulo-1-copa-mundial-de-la-fifatm.md  (2,792 bytes)
  articulo-002-articulo-2-fase-preliminar.md  (2,212 bytes)
  articulo-003-articulo-3-ente-organizador-de-la-fifa.md  (1,624 bytes)
  articulo-004-articulo-4-filiales-locales-de-la-fifa.md  (1,150 bytes)
  articulo-005-articulo-5-responsabilidades-de-las-federaciones-miembro-par.md  (5,739 bytes)
  articulo-006-articulo-6-retirada-partido-no-disputado-partido-suspendido-.md  (5,573 bytes)
  articulo-007-articulo-7-cuestiones-disciplinarias.md  (1,169 bytes)
  articulo-008-articulo-8-resolucion-de-disputas.md  (1,298 bytes)
  articulo-009-articulo-9-protestas.md  (4,002 bytes)
  ... y 43 más


In [13]:
# Tamaños: distribución de chars por archivo
tamanos = [f.stat().st_size for f in archivos]
if tamanos:
    print(f'Min:    {min(tamanos):,} bytes')
    print(f'Max:    {max(tamanos):,} bytes')
    print(f'Promedio: {sum(tamanos)//len(tamanos):,} bytes')
    print(f'Total:  {sum(tamanos):,} bytes')

Min:    605 bytes
Max:    24,986 bytes
Promedio: 3,025 bytes
Total:  160,347 bytes
